In [ ]:
%matplotlib widget

In [ ]:
from pathlib import Path
import numpy as np
import flammkuchen as fl
import pandas as pd
import tifffile as tiff

from fimpylab import TwoPExperiment
from split_dataset import SplitDataset

import matplotlib.pyplot as plt
import json

from scipy.interpolate import interp1d
from scipy import signal
from lotr.default_vals import REGRESSOR_TAU_S, TURN_BIAS


In [ ]:
def exp_decay_kernel(tau, dt, len_rec):
    upsample = 10
    t = np.arange(len_rec * upsample) * dt / upsample
    
    decay = np.exp(-t / tau)
    decay /= np.sum(decay)
    return decay

In [ ]:
master = Path(r"Z:\Hagar\e0075\only behavior\v13_front_cols_flashes_1x8")
master = Path(r"Z:\Hagar\e0075\only behavior\v12_front_cols_flashes_1x4")
files = list(master.glob("*_f*"))
files

In [ ]:
def scale_square(square_size, radius, proj_height):
    circumf = 2*np.pi*radius
    scale_factor = circumf / proj_height
    scaled_size = square_size * scale_factor
    return scaled_size

In [ ]:
num_rows = 4
n_regs = num_rows
s_size = 1 / num_rows
s_size = 0.5 / num_rows
radius = 0.3
proj_height = [2, 1.75, 1.5, 1.25, 1.25, 1.5, 1.75, 2]
proj_height = [2, 1.5, 1.5, 2]
        
        
choices = []
for x_pos in range(num_rows):
    scaled_size = scale_square(s_size, radius, proj_height[x_pos])
    curr_choice = [x_pos / num_rows, 0, scaled_size, 1]
    choices.append(curr_choice)

In [ ]:
choices

In [ ]:
metadata['stimulus']['protocol']['receptive_fields']

In [ ]:
for fish in files:
    print(fish)
    
    #try:
    metadata_file = list(fish.glob("*_metadata.json"))[0]

    with open(str(metadata_file), "r") as f:
        metadata = json.load(f)
    stim = metadata["stimulus"]["log"]


    pause_duration = stim[0]['duration']
    stim_duration = stim[1]['duration']


    n_options = num_rows
    n_rep = [metadata['stimulus']['protocol']['receptive_fields']['v12_front_cols_flashes_1x4']['n_trials']][0]
    n_trials = (n_options * n_rep) 
    position_list = np.zeros((n_trials, 4))    
    
    stim_timing = np.zeros((n_regs, n_rep)) # start time of each receptive field

    count = np.zeros((n_regs))
    for i in range(1, n_trials * 2, 2):
        curr_trial = stim[i]['clip_mask']
        position_list[(i//2) - 1, :] = curr_trial

        for j in range(n_options):
            if curr_trial == choices[j]:
                stim_timing[j, int(count[j])] = stim[i]['t_start']
                count[j] += 1

        d = {'pause_duration': pause_duration,
             'stim_duration': stim_duration,
             'stim_timings': stim_timing,
        }
        fl.save(fish / 'sensory_details.h5', d)
        
    #except:
    #    print("come on")

In [ ]:
len(stim)

In [ ]:
count